# SAR-LRA Tool for Google Colaboratory

This notebook facilitates the acquisition of Sentinel-1 SAR composite imagery for the VV_VH combination, encompassing both ascending and descending orbits, which can then be utilized to deploy the models.

Without modifications the notebook will predict in Taiwan.

### IMPORTANT:
Select a runtime with large RAM (such as TPU), otherwise the processing will brake.

It'll ask you to restart. You do not need to.

In [ ]:
# # install some libraries

# !pip install geedim
# !pip install leafmap==0.22.0
# !pip install geemap==0.24.1
# !pip install -U geemap
# !pip install earthengine-api
# !pip install rasterio
# !pip install opencv-python-headless
# !pip install requests
# !pip install tensorflow==2.12

In [1]:
import leafmap          
import geemap           
import ee               
import numpy as np      
import pandas as pd     
import rasterio
import tensorflow as tf
import numpy as np
import cv2
from utils import build_s1_rel_composites, run_lra_inference_per_rel

# Print versions
print("leafmap:", leafmap.__version__)           # Version 0.22.0
print("geemap:", geemap.__version__)             # Version 0.35.1
print("Earth Engine API:", ee.__version__)       # Version 1.5.3
print("numpy:", np.__version__)                  # Version 1.26.4
print("pandas:", pd.__version__)                 # Version 2.2.2
print("Rasterio version:", rasterio.__version__) # Version 1.4.3
print("TensorFlow version:", tf.__version__)     # Version 2.12.0
print("NumPy version:", np.__version__)          # Version 1.26.4
print("OpenCV version:", cv2.__version__)        # Version 4.11.0

leafmap: 0.28.1
geemap: 0.35.1
Earth Engine API: 1.4.3
numpy: 1.24.3
pandas: 2.0.3
Rasterio version: 1.3.6
TensorFlow version: 2.13.0
NumPy version: 1.24.3
OpenCV version: 4.7.0


## 2. Autenticate and initialize the Google Earth Engine

In [4]:
ee.Authenticate()
ee.Initialize(project='ee-lorenava-learning-sar') # replace with the name of your GEE project

## 3. Initialize and Define Area of Interest (AoI)

This section initializes a `Map` instance using the `geemap` library and centers it on a global view with a satellite basemap. The map will help define the Area of Interest (AOI).

### Instructions:
1. Use the map tools on the left to draw a single rectangular box defining your AOI.
2. Use the globe icon on the top left to search for specific locations if needed.



In [5]:
# Create a Map instance
Map = geemap.Map()

# Add a satellite basemap for visualization (similar to Google Earth)
Map.add_basemap('HYBRID')

# Center the map globally
Map.setCenter(0, 0, 2)  # Center on the world with zoom level 2

# Display the map for user interaction
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

## 4. Define temporal buffers and temporal stacks
Change the dates according to your event.

IMPORTANT: The landslides MUST be occurred in between the pre_end and post_start dates.

In [6]:
pre_stack_end = '2021-08-13'
post_stack_start = '2021-08-17'

build_s1_rel_composites(
    geometry=ee.FeatureCollection(Map.draw_features),     # your ee.Geometry / Feature / FeatureCollection
    pre_stack_end=pre_stack_end,       # "YYYY-MM-DD"
    post_stack_start=post_stack_start, # "YYYY-MM-DD"
    pre_days=60,
    post_days=12,
    orbits=("DESCENDING","ASCENDING"),
    project_path="",
    list_images=True,                  # set False to silence per-image prints
    fishnet_h=0.5, fishnet_v=0.5, fishnet_delta=1,
    scale=10, crs_epsg="EPSG:4326", error_margin_m=1,
    initialize_ee=False
)

SENTINEL 1 SAR IMAGE PROCESSING AND ACQUISITION !
_____________________________________________________________________________________
Setting time windows...
Pre stack start: 2021-06-14 00:00:00
Pre stack end  : 2021-08-13 00:00:00
Post stack start: 2021-08-17 00:00:00
Post stack end  : 2021-08-29 00:00:00

Orbit:  DESCENDING
project_path:  
training_path: deploy\VV_VH\60_12
outputs_path : outputs
_______________________________________________________________________________
-- PRE (DESCENDING): 5 items --
DATE         PLAT ORB  REL    INC(deg)   COV%AOI  ID
2021-06-16   S1A  DESCENDING 142          NA    100.00  S1A_IW_GRDH_1SDV_20210616T104733_20210616T104759_038364_0486FF_89E2
2021-06-28   S1A  DESCENDING 142          NA    100.00  S1A_IW_GRDH_1SDV_20210628T104734_20210628T104759_038539_048C42_B34D
2021-07-10   S1A  DESCENDING 142          NA    100.00  S1A_IW_GRDH_1SDV_20210710T104735_20210710T104800_038714_049184_AAA6
2021-07-22   S1A  DESCENDING 142          NA    100.00  S1A_

VV_VH_DESCENDING_REL142_1.tif: |          | 0.00/992M (raw) [  0.0%] in 00:00 (eta:     ?)

There is no STAC entry for: None


VV_VH_DESCENDING_REL142_2.tif: |          | 0.00/992M (raw) [  0.0%] in 00:00 (eta:     ?)

VV_VH_DESCENDING_REL142_3.tif: |          | 0.00/992M (raw) [  0.0%] in 00:00 (eta:     ?)

...

VV_VH_ASCENDING_REL77_3.tif: |          | 0.00/992M (raw) [  0.0%] in 00:00 (eta:     ?)

VV_VH_ASCENDING_REL77_4.tif: |          | 0.00/992M (raw) [  0.0%] in 00:00 (eta:     ?)

VV_VH_ASCENDING_REL77_5.tif: |          | 0.00/992M (raw) [  0.0%] in 00:00 (eta:     ?)

VV_VH_ASCENDING_REL77_6.tif: |          | 0.00/992M (raw) [  0.0%] in 00:00 (eta:     ?)

Downloaded 6 tiles in 149.77435398101807 seconds.
       Merged raster: deploy\VV_VH\60_12\ASCENDING\REL_77\SAR_ASCENDING_REL77.tif

DONE !!!


### Descending orbit

In [2]:
run_lra_inference_per_rel(
    place="Haiti",
    orbit="DESCENDING",
    base_dir="deploy/VV_VH/60_12",
    weights_url="https://github.com/lorenzonava96/SAR-and-DL-for-Landslide-Rapid-Assessment/raw/main/SAR-LRA%20Tool%20V1/model_weights/VV_VH_60_nn_noSlope_DESCENDING_60_12_5_size_64_filters_64_batch_size_512_lr_0.001_dropout_0.7_fil1_3_fil2_3_fil3_3.hdf5",
    # or weights_local_path="path/to/local_descending_weights.hdf5",
    size=64,
    channels=4,
    prob_thresh=0.6,
    nms_overlap=0.1,
    batch_size=512,
    filtersFirstLayer=64
)

[INFO] initializing model...
[INFO] downloading weights for DESCENDING ...
[INFO] weights saved to model_weights_descending.hdf5
[INFO] loading model weights...
[INFO] model ready.
[INFO] found 1 REL tracks for DESCENDING: [142]

[INFO] loading image for DESCENDING REL 142:
       deploy/VV_VH/60_12\DESCENDING\REL_142\SAR_DESCENDING_REL142.tif
[INFO] extracted 179,920 patches in 16.532s
[INFO] classifying ROIs...
[INFO] classification done in 167.241s
[INFO] kept 340 boxes after NMS
[INFO] wrote raster mask: predictions\Haiti_DESCENDING_REL142.tif
[INFO] vectorizing detections...
[INFO] wrote shapefile: predictions\Haiti_DESCENDING_REL142.tif.shp

[INFO] done with orbit: DESCENDING


### Ascending orbit

In [3]:
run_lra_inference_per_rel(
    place="Haiti",
    orbit="ASCENDING",
    base_dir="deploy/VV_VH/60_12",
    weights_url="https://github.com/lorenzonava96/SAR-and-DL-for-Landslide-Rapid-Assessment/raw/main/SAR-LRA%20Tool%20V1/model_weights/VV_VH_60_nn_noSlope_ASCENDING_60_12_6_size_64_filters_32_batch_size_512_lr_0.001_dropout_0.7_fil1_3_fil2_3_fil3_3.hdf5",
    # or weights_local_path="path/to/local_descending_weights.hdf5",
    size=64,
    channels=4,
    prob_thresh=0.6,
    nms_overlap=0.1,
    batch_size=512,
    filtersFirstLayer=32
)

[INFO] initializing model...
[INFO] downloading weights for ASCENDING ...
[INFO] weights saved to model_weights_ascending.hdf5
[INFO] loading model weights...
[INFO] model ready.
[INFO] found 2 REL tracks for ASCENDING: [4, 77]

[INFO] loading image for ASCENDING REL 4:
       deploy/VV_VH/60_12\ASCENDING\REL_4\SAR_ASCENDING_REL4.tif
[INFO] extracted 179,920 patches in 16.306s
[INFO] classifying ROIs...
[INFO] classification done in 82.491s
[INFO] kept 118 boxes after NMS
[INFO] wrote raster mask: predictions\Haiti_ASCENDING_REL4.tif
[INFO] vectorizing detections...
[INFO] wrote shapefile: predictions\Haiti_ASCENDING_REL4.tif.shp

[INFO] loading image for ASCENDING REL 77:
       deploy/VV_VH/60_12\ASCENDING\REL_77\SAR_ASCENDING_REL77.tif
[INFO] extracted 179,920 patches in 16.969s
[INFO] classifying ROIs...
[INFO] classification done in 84.696s
[INFO] kept 125 boxes after NMS
[INFO] wrote raster mask: predictions\Haiti_ASCENDING_REL77.tif
[INFO] vectorizing detections...
[INFO] wrote 